# 05 · Wind × runway features per airport-hour
For each hour and each runway end (from 02_nasr):
- headwind = W·cos(wind_dir − heading)
- crosswind = |W·sin(wind_dir − heading)|

W is the sustained wind, or the gust if that's higher.

An end is **usable** if crosswind ≤ `MAX_CROSSWIND_KT` and tailwind (negative headwind) ≤ `MAX_TAILWIND_KT`. Calm (< `CALM_WIND_KT`) or variable wind makes every end usable.

Wind direction (GHCNh) and runway heading (`TRUE_ALIGNMENT`) are both relative to **true** north, so no magnetic correction is needed.

Output: `data/interim/runway_wind_{APT}.parquet`

In [1]:
%run ../config.ipynb

ROOT = /Users/dylannegvesky/Documents/FSU Documents/Fall 2026/ProjDS/PDS_Final
study airports: ['EWR'] | BTS 2023-01 → latest | train ≤ 2025-12


In [2]:
runway_ends = pd.read_parquet(INTERIM / "runway_ends.parquet")


def runway_wind_features(wx, ends):
    hdg = ends["true_hdg"].to_numpy()                                   # (E,)
    ils = ends["has_ils"].to_numpy()
    phys = pd.factorize(ends["physical_runway"])[0]
    heading_group = np.round((hdg % 180) / 10).astype(int) % 18           # parallel runways share a group

    wind = np.fmax(wx["wind_kt"].to_numpy(), wx["gust_kt"].fillna(0).to_numpy())[:, None]   # (H,1)
    wdir = wx["wind_dir_deg"].to_numpy()[:, None]
    rel = np.radians(wdir - hdg[None, :])                                  # (H,E)
    head = wind * np.cos(rel)
    cross = np.abs(wind * np.sin(rel))

    calm_or_var = (wind < CALM_WIND_KT) | np.isnan(wdir)
    usable = ((cross <= MAX_CROSSWIND_KT) & (head >= -MAX_TAILWIND_KT)) | calm_or_var
    missing = np.isnan(wind[:, 0])

    out = pd.DataFrame({"hour_utc": wx["hour_utc"]})
    out["n_ends"] = len(hdg)
    out["n_usable_ends"] = usable.sum(1)
    out["n_usable_ils_ends"] = (usable & ils[None, :]).sum(1)
    out["frac_usable"] = out["n_usable_ends"] / len(hdg)
    # Physical runways / heading groups with at least one usable end
    out["n_usable_physical"] = [len(set(phys[row])) for row in usable]
    out["n_usable_headings"] = [len(set(heading_group[row])) for row in usable]
    # Best-case wind on the most favourable end
    cross_c = np.where(calm_or_var, 0.0, cross)
    head_c = np.where(calm_or_var, 0.0, head)
    out["min_crosswind_kt"] = cross_c.min(1)
    out["max_headwind_kt"] = head_c.max(1)
    out["crosswind_on_best_kt"] = cross_c[np.arange(len(cross_c)), head_c.argmax(1)]
    # Arrival runways in instrument conditions: only ILS ends count when IMC
    imc = wx["imc"].fillna(0).to_numpy().astype(bool)
    out["n_usable_arrival_ends"] = np.where(imc, out["n_usable_ils_ends"], out["n_usable_ends"])
    feat_cols = [c for c in out if c not in ("hour_utc", "n_ends")]
    out.loc[missing, feat_cols] = np.nan
    return out

In [3]:
rw = {}
for apt in STUDY_AIRPORTS:
    wx = pd.read_parquet(INTERIM / f"weather_{apt}.parquet")
    ends = runway_ends[runway_ends["faa_id"] == to_faa(apt)].reset_index(drop=True)
    assert len(ends), f"no runway ends for {apt}"
    feats = runway_wind_features(wx, ends)
    feats.insert(0, "airport", apt)
    rw[apt] = feats
    feats.to_parquet(INTERIM / f"runway_wind_{apt}.parquet", index=False)
    print(f"{apt}: {len(ends)} runway ends, {len(feats):,} hours")
    display(ends[["end_id", "true_hdg", "has_ils"]].T)

EWR: 6 runway ends, 34,672 hours


,0,1,2,3,4,5
end_id,04L,22R,04R,22L,11,29
true_hdg,26.0,206.0,26.0,206.0,95.0,275.0
has_ils,True,True,True,True,True,False


## Checks

In [4]:
for apt, feats in rw.items():
    wx = pd.read_parquet(INTERIM / f"weather_{apt}.parquet")
    print(f"== {apt}: distribution of usable ends")
    display(feats["n_usable_ends"].value_counts(normalize=True).sort_index().round(3).to_frame().T)
    # A strong crosswind for every runway should leave few usable ends
    windy = wx["wind_kt"].fillna(0).clip(lower=wx["gust_kt"].fillna(0)) >= 25
    print("mean usable ends when wind/gust ≥ 25 kt:", feats.loc[windy, "n_usable_ends"].mean().round(2),
          "| overall:", feats["n_usable_ends"].mean().round(2))
    print("hours with no usable end:", (feats["n_usable_ends"] == 0).sum())
    display(feats.merge(wx[["hour_utc", "wind_dir_deg", "wind_kt", "gust_kt", "imc"]], on="hour_utc")
            .loc[windy].head(8))

== EWR: distribution of usable ends


n_usable_ends,0.0,1.0,2.0,3.0,4.0,5.0,6.0
proportion,0.011,0.07,0.022,0.201,0.221,0.149,0.327


mean usable ends when wind/gust ≥ 25 kt: 1.17 | overall: 4.31
hours with no usable end: 390


,airport,hour_utc,n_ends,n_usable_ends,n_usable_ils_ends,frac_usable,n_usable_physical,n_usable_headings,min_crosswind_kt,max_headwind_kt,crosswind_on_best_kt,n_usable_arrival_ends,wind_dir_deg,wind_kt,gust_kt,imc
0,EWR,2022-12-01 00:00:00+00:00,6,1.0,0.0,0.166667,1.0,1.0,8.049662,30.041746,8.049662,1.0,260.0,25.075588,31.101504,0.0
1,EWR,2022-12-01 01:00:00+00:00,6,1.0,0.0,0.166667,1.0,1.0,2.270190,25.948391,2.270190,1.0,270.0,13.995677,26.047510,0.0
6,EWR,2022-12-01 06:00:00+00:00,6,1.0,0.0,0.166667,1.0,1.0,2.185481,24.980167,2.185481,1.0,270.0,17.105827,25.075588,0.0
7,EWR,2022-12-01 07:00:00+00:00,6,1.0,0.0,0.166667,1.0,1.0,2.354899,26.916615,2.354899,1.0,280.0,18.077749,27.019432,0.0
16,EWR,2022-12-01 16:00:00+00:00,6,1.0,0.0,0.166667,1.0,1.0,2.270190,25.948391,2.270190,1.0,280.0,17.105827,26.047510,0.0
17,EWR,2022-12-01 17:00:00+00:00,6,1.0,0.0,0.166667,1.0,1.0,6.741592,25.159962,6.741592,1.0,290.0,20.021593,26.047510,0.0
64,EWR,2022-12-03 16:00:00+00:00,6,2.0,2.0,0.333333,2.0,1.0,2.722706,25.904819,2.722706,2.0,200.0,14.967599,26.047510,1.0
65,EWR,2022-12-03 17:00:00+00:00,6,2.0,2.0,0.333333,2.0,1.0,1.749185,25.014505,1.749185,2.0,210.0,15.939521,25.075588,1.0
